In [2]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
DATA_DIR = '/kaggle/input/competitions/domestic-flight-price-prediction-challenge'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

test_ids = test['id'].copy()

train = train.dropna(subset=['price']).reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain null counts:\n", train.isnull().sum())
print("\nTest null counts:\n", test.isnull().sum())

# ------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------
STOP_MAP = {
    'zero': 0,
    'one': 1,
    'two_or_more': 2
}

CLASS_MAP = {
    'Economy': 0,
    'Business': 1
}

# Original categorical columns
BASE_CAT_COLS = [
    'airline',
    'source_city',
    'destination_city',
    'departure_time',
    'arrival_time',
    'flight'
]


def get_prefix(df):
    return df['flight'].astype('string').str.split().str[0]


# ------------------------------------------------------------------
# 3. Learn imputation values from training data only
# ------------------------------------------------------------------
def fit_preprocessor(df):
    p = {}

    # Flight prefix -> most common airline
    tmp = pd.DataFrame({
        'prefix': get_prefix(df),
        'airline': df['airline']
    }).dropna()

    p['airline_map'] = (
        tmp.groupby('prefix')['airline']
        .agg(lambda s: s.mode().iloc[0])
        .to_dict()
    )

    # Impute stops and class
    stops = df['stops'].map(STOP_MAP)
    p['stops_mode'] = stops.mode().iloc[0]

    classes = df['flight_class'].map(CLASS_MAP)
    p['class_mode'] = classes.mode().iloc[0]

    stops = stops.fillna(p['stops_mode'])

    # Duration median by stops group
    p['duration_by_stops'] = (
        df.assign(stops_num=stops)
        .groupby('stops_num')['duration']
        .median()
        .to_dict()
    )

    p['duration_median'] = df['duration'].median()
    p['days_left_median'] = df['days_left'].median()

    # Categorical fallback values
    p['cat_modes'] = {
        c: df[c].mode().iloc[0]
        for c in BASE_CAT_COLS
    }

    return p


# ------------------------------------------------------------------
# 4. Clean data and create features
# ------------------------------------------------------------------
def clean(df, p):
    df = df.copy()

    # Fill airline using flight prefix, then mode
    prefix = get_prefix(df)

    df['airline'] = df['airline'].fillna(
        prefix.map(p['airline_map'])
    )
    df['airline'] = df['airline'].fillna(
        p['cat_modes']['airline']
    )

    # Numeric encoding for stops and class
    df['stops'] = (
        df['stops'].map(STOP_MAP)
        .fillna(p['stops_mode'])
    )

    df['flight_class'] = (
        df['flight_class'].map(CLASS_MAP)
        .fillna(p['class_mode'])
    )

    # Impute duration
    df['duration'] = df['duration'].fillna(
        df['stops'].map(p['duration_by_stops'])
    )
    df['duration'] = df['duration'].fillna(
        p['duration_median']
    )

    # Impute days_left
    df['days_left'] = df['days_left'].fillna(
        p['days_left_median']
    )

    # Fill categorical columns
    for col, mode_value in p['cat_modes'].items():
        df[col] = df[col].fillna(mode_value)

    # Extract flight prefix as a lower-cardinality feature.
    # We will drop the full flight identifier before encoding.
    df['flight_prefix'] = get_prefix(df).fillna('Unknown').astype(str)

    # Feature engineering: route
    df['route'] = (
        df['source_city'].astype(str)
        + '_to_'
        + df['destination_city'].astype(str)
    )

    # Feature engineering: airline + route
    df['airline_route'] = (
        df['airline'].astype(str)
        + '_on_'
        + df['route']
    )

    # Nonlinear booking-time feature
    df['days_left_squared'] = df['days_left'] ** 2

    # Duration and stops interaction
    df['duration_x_stops'] = (
        df['duration'] * df['stops']
    )

    # Booking time and business-class interaction
    df['days_left_x_business'] = (
        df['days_left']
        * (df['flight_class'] == 1).astype(int)
    )

    # Drop identifier and full flight code to avoid a huge
    # one-hot feature space for individual flight numbers.
    df = df.drop(
        columns=['id'],
        errors='ignore'
    )

    return df


params = fit_preprocessor(train)

train_clean = clean(train, params)
test_clean = clean(test, params)

y = train_clean['price'].copy()

X_all = train_clean.drop(columns=['price'])
X_test_all = test_clean.copy()

# ------------------------------------------------------------------
# 5. One-hot encode categorical columns
# ------------------------------------------------------------------
CAT_COLS = [
    'airline',
    'source_city',
    'destination_city',
    'departure_time',
    'arrival_time',
    'flight_prefix',
    'route',
    'airline_route',
    'flight'
]

# Generate dummy columns independently, then align test to train
X_enc = pd.get_dummies(
    X_all,
    columns=CAT_COLS,
    dtype=np.uint8
)

X_test_enc = pd.get_dummies(
    X_test_all,
    columns=CAT_COLS,
    dtype=np.uint8
)

X_test_enc = X_test_enc.reindex(
    columns=X_enc.columns,
    fill_value=0
)

# Check consistency
assert list(X_enc.columns) == list(X_test_enc.columns)
assert X_enc.isnull().sum().sum() == 0
assert X_test_enc.isnull().sum().sum() == 0

print("\nEncoded train shape:", X_enc.shape)
print("Encoded test shape:", X_test_enc.shape)

# ------------------------------------------------------------------
# 6. Train/validation split
# ------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_enc,
    y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------------------------------
# 7. Train XGBoost
# ------------------------------------------------------------------
model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    tree_method='hist',
    early_stopping_rounds=150,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=200
)

# ------------------------------------------------------------------
# 8. Validation metrics
# ------------------------------------------------------------------
y_pred = model.predict(X_val)

print("\n--- XGBoost Validation Performance ---")
print(f"MAE : {mean_absolute_error(y_val, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred)):.2f}")
print(f"R2  : {r2_score(y_val, y_pred):.4f}")

print("Best iteration:", model.best_iteration)

# ------------------------------------------------------------------
# 9. Feature importance
# ------------------------------------------------------------------
importance = pd.DataFrame({
    'feature': X_enc.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 20 Features ---")
print(importance.head(20).to_string(index=False))

# ------------------------------------------------------------------
# 10. Train final model using all training data
# ------------------------------------------------------------------
# Use the best iteration selected on the validation split.
best_n_estimators = model.best_iteration + 1

final_model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    n_estimators=best_n_estimators,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

final_model.fit(X_enc, y, verbose=False)

# ------------------------------------------------------------------
# 11. Predict test and save submission
# ------------------------------------------------------------------
test_pred = final_model.predict(X_test_enc)

# Prices cannot be negative
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    'id': test_ids,
    'price': test_pred
})

submission.to_csv('submission_xgboost.csv', index=False)

print("\nSaved submission_xgboost.csv")
print(submission.shape)
print(submission.head())


Train shape: (15589, 12)
Test shape: (3898, 11)

Train null counts:
 id                     0
airline             3089
flight                 0
source_city            0
departure_time         0
stops                  0
arrival_time           0
destination_city       0
flight_class           0
duration            3095
days_left              0
price                  0
dtype: int64

Test null counts:
 id                    0
airline             799
flight                0
source_city           0
departure_time        0
stops                 0
arrival_time          0
destination_city      0
flight_class          0
duration            776
days_left             0
dtype: int64

Encoded train shape: (15589, 1406)
Encoded test shape: (3898, 1406)
[0]	validation_0-rmse:35848.12115
[200]	validation_0-rmse:9651.95183
[400]	validation_0-rmse:9610.68870
[464]	validation_0-rmse:9614.21066

--- XGBoost Validation Performance ---
MAE : 3572.68
RMSE: 9601.62
R2  : 0.9346
Best iteration: 314

--- Top 20 